# Financial Market Regime Detection

Detect market synchronization regimes using the Kuramoto order parameter.
R(t) → 1 precedes crashes (Black Monday 1987, 2008 crisis).

This notebook:
1. Generates synthetic market data with a herding event
2. Extracts phase via Hilbert transform
3. Computes R(t) and detects regime transitions
4. Identifies crash early warning signals

In [ ]:
import numpy as np
from scpn_phase_orchestrator.upde.market import (
    extract_phase,
    market_order_parameter,
    detect_regimes,
    sync_warning,
    market_plv,
)

## Generate Synthetic Market Data

Three phases: normal trading → herding (correlated) → aftermath

In [ ]:
rng = np.random.default_rng(42)
N_ASSETS = 10

# Phase 1: Normal (independent)
normal = rng.standard_normal((200, N_ASSETS))

# Phase 2: Herding (all assets follow same signal)
base = np.sin(np.linspace(0, 8 * np.pi, 150))
herding = np.column_stack(
    [base + 0.3 * rng.standard_normal(150) for _ in range(N_ASSETS)]
)

# Phase 3: Aftermath (back to independent)
aftermath = rng.standard_normal((150, N_ASSETS))

returns = np.vstack([normal, herding, aftermath])
print(f"Data: {returns.shape[0]} timesteps × {returns.shape[1]} assets")

## Phase Extraction & Order Parameter

In [ ]:
phases = extract_phase(returns)
R = market_order_parameter(phases)

print(f"Mean R (normal t=0-199):    {np.mean(R[:200]):.3f}")
print(f"Mean R (herding t=200-349): {np.mean(R[200:350]):.3f}")
print(f"Mean R (aftermath t=350+):  {np.mean(R[350:]):.3f}")

## Regime Detection & Early Warning

In [ ]:
regimes = detect_regimes(R, sync_threshold=0.7, desync_threshold=0.3)
warnings = sync_warning(R, threshold=0.6, lookback=10)

names = {0: "DESYNC", 1: "TRANSITION", 2: "SYNCHRONIZED"}
for label in [0, 1, 2]:
    count = np.sum(regimes == label)
    print(
        f"  {names[label]:>14s}: {count:3d} timesteps ({100 * count / len(regimes):.1f}%)"
    )

print(f"\nSync warnings: {np.sum(warnings)}")
warning_times = np.where(warnings)[0]
if len(warning_times) > 0:
    print(f"Warning at timesteps: {warning_times.tolist()}")

## Pairwise PLV Matrix (Windowed)

In [ ]:
plv = market_plv(phases, window=50)
print(f"PLV shape: {plv.shape}")

# Compare PLV during normal vs herding
plv_normal = np.mean(plv[:150], axis=0)  # first 150 windows ≈ normal
plv_herding = np.mean(plv[150:300], axis=0)  # windows during herding

# Mean off-diagonal PLV
mask = ~np.eye(N_ASSETS, dtype=bool)
print(f"Mean PLV (normal):  {np.mean(plv_normal[mask]):.3f}")
print(f"Mean PLV (herding): {np.mean(plv_herding[mask]):.3f}")